# 03 · Part B: school-grouped folds and the in-fold pipeline

Part A built one clean table using **fixed rules**. Part B is everything that **learns numbers from the data**
(the median used to fill a blank, the mean and spread used for scaling, the clipping limits),
plus the train/test split those steps have to respect.
In the project plan it is **T07** (the fold file) and **T08** (the preprocessing pipeline). In the brief it counts towards
**R2** ("comprehensive data preprocessing and cleaning"), and in the report it belongs in *Experimental Setup*.

The code lives in one shared file, `02_data-analysis/pisa_prep.py`, so every model in R3 and R4 uses exactly the same folds,
preprocessing, targets and weights. This notebook builds the fold file and then **checks** the folds and the pipeline:
no school on both sides of a split, folds that look alike, and every learned number coming from the training schools only.

| | |
|---|---|
| **Reads** | `data/clean/pisa6_clean.parquet`, `data/clean/columns.json` (from Part A) |
| **Writes** | `data/clean/folds.csv` (which fold each school is in) |
| **Audit files** | `02_data-analysis/outputs/part_b/` (fold balance, the learned-numbers check, the smoke test) |
| **Uses** | `02_data-analysis/pisa_prep.py` (the shared Part B code) |
| **Run time** | under a minute |

The sections:

1. Setup
2. Split the schools into 5 folds
3. Check the folds
4. Choose the model inputs
5. The preprocessing pipeline
6. Proof that the pipeline only learns from the training schools
7. Weights
8. Smoke test: does a simple model run end to end?
9. How to use Part B in your own notebook
10. Settings the group still has to agree on

> **AI-use note.** The code in this notebook and in `pisa_prep.py` was generated by an AI assistant (Claude Code, Anthropic, model Claude Opus 5.5) on 17 Nov 2026.
> Following the coursework rule, here are the full prompts:
>
> 1. *"Forget about the branches by fatah and annie. I want to preform the preprocessing, cleanup, and the EDA independent of their work... If we want to perform preprocessing and want to cleanup this data, what are some actions we should take? Assume Im a noob at all of this and need explanations on terminologies and approaches."* The answer split the work into Part A (fixed rules) and Part B (steps that learn from the data).
> 2. *"can you explain part B? where is that in the coursework?"*
> 3. *"Yes, build Part B"*, answering the question "Should I build Part B (the fold file and the in-fold pipeline)?".
>
> The defaults for the four open team decisions (section 10) were chosen by the AI and are marked as defaults in `pisa_prep.py`, so the group can review and change them.

## How the variable names work

Python doesn't say what kind of thing a variable holds, so the **names** do. The same rules are used in every notebook in this folder.

| The name... | means it holds | Examples |
|---|---|---|
| is a singular word | **one** value | `system` = `"ARE"`, `feature` = `"ESCS"` |
| is a plural word | a **list** of values | `SYSTEMS` = `["ARE", "SAU", ...]` |
| ends in `_col` | **one** column name | `pv_col` = `"PV1MATH"`, `weight_col` = `"W_FSTUWT"` |
| ends in `_cols` | a **list** of column names | `PV_COLS` = `["PV1MATH", ..., "PV10MATH"]`, `FEATURE_COLS` |
| ends in `_df` | a **table** (a pandas DataFrame) | `students_df`, `system_df` (one system's students) |
| is just `df` | **the main table** of the notebook | |
| looks like `<value>_by_<key>` | a **lookup**: give it a key, get a value (a dict, or a pandas Series) | `FEATURES_BY_GROUP["family"]` gives the 4 family columns |
| starts with `is_` or `has_` | **True/False for each student**, used as a filter | `df[is_complete]` keeps the students with no blanks |
| starts with `n_` / `pct_` / `share_` | a count / a percentage (0-100) / a fraction (0-1) | `n_flagged_students`, `pct_missing_df`, `share_label_flips` |
| ends in `_idx` | a position (0, 1, 2, ...) | `row_idx` |
| ends in `_DIR` | a folder | `CLEAN_DIR`, `FIGURE_DIR` |
| ends in `_COLOR` / `_CMAP` | one colour / a colour scale | `TEXT_COLOR`, `BLUES_CMAP` |
| is in `UPPER_CASE` | a **fixed setting**, set once and never changed | `LEVEL2_CUTOFF`, `SYSTEMS` |
| is `fig`, `ax` or `axes` | a whole figure / one chart panel / several panels | matplotlib's usual names |

Loops use the singular of the list they walk through: `for system in SYSTEMS`, `for feature in FEATURE_COLS`.
Functions are named after what they return (`weighted_quantile()`) or what they do (`log_step()`, `draw_heatmap()`).
Each helper function has **type hints** saying what goes in and what comes out, e.g. `cols: list[str]` (a list of column names) and `-> float` (returns one number).

The column names *inside* the data (`ESCS`, `PV1MATH`, `W_FSTUWT`) are PISA's own. We don't rename them, so they still match the OECD documentation.

## 1. Setup

The shared code is imported as `prep`. Everything it provides is listed in section 9.
`REBUILD_FOLDS` stays `False`: the fold file is made once and then frozen, like the clean table (section 2 explains why).

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import r2_score, roc_auc_score, root_mean_squared_error

ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "requirements.txt").exists())
sys.path.insert(0, str(ROOT / "02_data-analysis"))
import pisa_prep as prep   # the shared Part B code

OUTPUT_DIR = ROOT / "02_data-analysis/outputs/part_b"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REBUILD_FOLDS = False      # True = replace the fold file. Every model's results would change, so agree it with the group first.
pd.set_option("display.width", 200, "display.max_columns", 40, "display.float_format", "{:,.3f}".format)

NAME_BY_SYSTEM = {"ARE": "UAE", "SAU": "Saudi Arabia", "QAT": "Qatar", "JOR": "Jordan", "MAR": "Morocco", "PSE": "Palestine"}

def weighted_mean_over_pvs(table_df: pd.DataFrame, cols: list[str], weight_col: str) -> float:
    """Weighted mean of each column in cols (the 10 PVs, or the 10 below-Level-2 labels), then the average of those means."""
    return np.mean([np.average(table_df[col], weights=table_df[weight_col]) for col in cols])

df = prep.load_clean_table()
print(f"{len(df):,} students, {df['CNTSCHID'].nunique():,} schools, {df.shape[1]} columns")
print(f"Folds: {prep.N_FOLDS}, seed {prep.FOLD_SEED}")
print(f"Team settings: POOLED_WEIGHT_COL={prep.POOLED_WEIGHT_COL!r}, USE_SYSTEM_AS_FEATURE={prep.USE_SYSTEM_AS_FEATURE}, "
      f"FAMILY_AS_ESCS_ONLY={prep.FAMILY_AS_ESCS_ONLY}, CLIP_PERCENTILES={prep.CLIP_PERCENTILES}")

61,775 students, 1,973 schools, 151 columns
Folds: 5, seed 2022
Team settings: POOLED_WEIGHT_COL='SENWT', USE_SYSTEM_AS_FEATURE=False, FAMILY_AS_ESCS_ONLY=False, CLIP_PERCENTILES=(1.0, 99.0)


## 2. Split the schools into 5 folds

**Why test on data the model hasn't seen.** A model always looks better on the students it learned from.
To get an honest score we hide some students, train on the rest, and score the model on the hidden ones.

**Why 5 folds (cross-validation).** One split would give one score that depends on which students happened to be hidden.
Instead we cut the data into 5 parts, called **folds**. Each round, one fold is the **test** set and the other four are the
**training** set. After 5 rounds every student has been tested exactly once, and we get 5 scores instead of one.

**Why whole schools.** Students in the same school are alike: same teachers, same neighbourhood, similar families.
The EDA (section 9) found that 16-47% of the variation in maths scores lies *between* schools.
If a school had students on both sides, the model could partly recognise the school instead of learning about students,
and its test score would be too good. Our success criterion is AUC on **unseen schools**, so each school goes into exactly one fold.
This is the "group" part of `StratifiedGroupKFold`.

**Why stratified.** We also want the folds to look alike: every system in every fold, and a similar share of students below Level 2.
Each student gets a label like `MAR_below` (system + whether most of their 10 plausible values are below Level 2),
and the splitter spreads each label evenly across the folds. This is the "stratified" part.

**Why a file.** The split is saved to `data/clean/folds.csv` (one row per school) and everyone loads that file.
If each person made their own folds, differences between models could just be differences between splits.
This notebook only makes a new file if there isn't one yet (or if `REBUILD_FOLDS = True`).

In [2]:
if prep.FOLDS_PATH.exists() and not REBUILD_FOLDS:
    folds_df = prep.load_folds()
    print(f"Loaded the frozen fold file {prep.FOLDS_PATH.relative_to(ROOT)}")
else:
    folds_df = prep.make_folds(df)
    prep.save_folds(folds_df)
    print(f"Made and saved a new fold file {prep.FOLDS_PATH.relative_to(ROOT)}")

# Does the code still make the same folds? (A different scikit-learn version might not. The saved file is what counts.)
rebuilt_folds_df = prep.make_folds(df)
print("Rebuilding the folds today gives the same file:", rebuilt_folds_df.equals(folds_df))

fold_by_student = prep.get_fold_by_student(df, folds_df)   # which test fold (0-4) each student is in
folds_df.head()

Loaded the frozen fold file data\clean\folds.csv


Rebuilding the folds today gives the same file: True


,CNTSCHID,CNT,fold,n_students
0,27500001,PSE,2,10
1,27500002,PSE,4,35
2,27500003,PSE,0,35
3,27500004,PSE,3,15
4,27500005,PSE,1,33


## 3. Check the folds

Four checks that must all pass. If any fails, the notebook stops here with an error.

In [3]:
# 1. Every school is in exactly one fold, and every school in the table is in the file.
assert folds_df["CNTSCHID"].is_unique
assert set(folds_df["CNTSCHID"]) == set(df["CNTSCHID"])

# 2. All the students of a school are in their school's fold.
assert (df.assign(fold=fold_by_student).groupby("CNTSCHID")["fold"].nunique() == 1).all()

# 3. In every round, no school has students on both the training and the test side.
for fold in range(prep.N_FOLDS):
    is_test = fold_by_student == fold
    train_school_ids = set(df.loc[~is_test, "CNTSCHID"])
    test_school_ids = set(df.loc[is_test, "CNTSCHID"])
    assert not train_school_ids & test_school_ids, f"fold {fold}: a school is on both sides"

# 4. Every system has schools in every fold.
n_schools_df = pd.crosstab(folds_df["CNT"], folds_df["fold"]).loc[prep.SYSTEMS]
assert (n_schools_df > 0).all().all()

print("All 4 checks passed.")

All 4 checks passed.


Do the folds look alike? For each fold: how many schools and students it has, the % of students below Level 2,
and the average maths score. The last two are weighted with `SENWT` (each system counts equally) and averaged over the 10 plausible values.

In [4]:
balance_rows = []   # one dict per fold
for fold in range(prep.N_FOLDS):
    fold_df = df[fold_by_student == fold]
    balance_rows.append({
        "fold": fold,
        "n_schools": fold_df["CNTSCHID"].nunique(),
        "n_students": len(fold_df),
        "pct_students": 100 * len(fold_df) / len(df),
        "pct_below_l2": 100 * weighted_mean_over_pvs(fold_df, prep.BELOW_L2_COLS, "SENWT"),
        "mean_maths": weighted_mean_over_pvs(fold_df, prep.PV_COLS, "SENWT"),
    })
balance_df = pd.DataFrame(balance_rows).set_index("fold")
balance_df.to_csv(OUTPUT_DIR / "fold_balance.csv")
balance_df.style.format({"n_schools": "{:,}", "n_students": "{:,}", "pct_students": "{:.1f}",
                         "pct_below_l2": "{:.1f}", "mean_maths": "{:.1f}"})

,n_schools,n_students,pct_students,pct_below_l2,mean_maths
fold,,,,,
0,397,"12,340",20.0,69.3,389.6
1,396,"12,330",20.0,69.6,387.0
2,394,"12,365",20.0,71.2,384.5
3,394,"12,361",20.0,69.6,388.5
4,392,"12,379",20.0,70.1,388.6


The same check for each system: the number of schools in each fold, and the % below Level 2 in each fold
(weighted with `W_FSTUWT`, the usual weight inside one system).

In [5]:
pct_below_rows = []   # one dict per system
for system in prep.SYSTEMS:
    system_df = df[df["CNT"] == system]
    system_fold_by_student = fold_by_student[system_df.index]
    row = {"system": NAME_BY_SYSTEM[system]}
    for fold in range(prep.N_FOLDS):
        row[f"fold {fold}"] = 100 * weighted_mean_over_pvs(system_df[system_fold_by_student == fold], prep.BELOW_L2_COLS, "W_FSTUWT")
    pct_below_rows.append(row)
pct_below_by_system_df = pd.DataFrame(pct_below_rows).set_index("system")
pct_below_by_system_df["largest gap"] = pct_below_by_system_df.max(axis=1) - pct_below_by_system_df.min(axis=1)

n_schools_by_system_df = n_schools_df.rename(index=NAME_BY_SYSTEM, columns=lambda fold: f"fold {fold}")
n_schools_by_system_df.to_csv(OUTPUT_DIR / "fold_schools_by_system.csv")
pct_below_by_system_df.to_csv(OUTPUT_DIR / "fold_pct_below_l2_by_system.csv")
print("Schools per fold:")
display(n_schools_by_system_df)
print("% below Level 2 per fold:")
display(pct_below_by_system_df.style.format("{:.1f}"))

Schools per fold:


fold,fold 0,fold 1,fold 2,fold 3,fold 4
CNT,,,,,
UAE,167,169,167,168,169
Saudi Arabia,38,39,39,39,38
Qatar,47,46,46,45,45
Jordan,54,52,52,52,50
Morocco,37,36,35,35,35
Palestine,54,54,55,55,55


% below Level 2 per fold:


,fold 0,fold 1,fold 2,fold 3,fold 4,largest gap
system,,,,,,
UAE,46.2,49.9,50.1,49.7,49.3,3.9
Saudi Arabia,69.8,69.7,69.4,69.9,71.4,2.0
Qatar,56.9,55.1,59.9,54.8,56.5,5.1
Jordan,82.0,82.5,84.2,83.0,82.6,2.2
Morocco,79.0,82.4,82.3,82.1,82.2,3.4
Palestine,81.5,81.2,79.2,80.0,78.0,3.5


**So what.** The five folds are almost identical in size (392-397 schools, 20% of the students each) and in difficulty:
pooled, 69-71% of students are below Level 2 in every fold and the average maths score is 385-390.
Inside each system, the folds differ by 2-5 points in % below Level 2, most in Qatar, which only has 45-47 schools per fold.
So if a model scores differently on different folds, that reflects the schools in them, not a lopsided split.
The rebuild check also shows the split can be reproduced with this version of scikit-learn.

## 4. Choose the model inputs

The model inputs are the pitch's **17 features** plus one **missing flag** per feature (`ESCS_missing` = 1 if the student's ESCS was blank).
Part A made the flags. The EDA showed that being blank goes with different scores, so the flags carry real information,
and they let the model tell a filled-in value from a real one.

`prep.get_feature_cols()` returns the columns for the factor groups you ask for. A group's missing flags come and go with the group,
so an **ablation** ("how good is the model without the classroom group?") removes the group's features *and* its flags.

In [6]:
feature_rows = []   # one dict per feature
for group in prep.GROUPS:
    for feature in prep.FEATURES_BY_GROUP[group]:
        feature_rows.append({"group": group, "feature": feature,
                             "kind": "0/1" if feature in prep.BINARY_FEATURE_COLS else "numeric",
                             "missing_flag": f"{feature}_missing", "pct_blank": 100 * df[feature].isna().mean()})
feature_table_df = pd.DataFrame(feature_rows)

feature_cols = prep.get_feature_cols()   # all 5 groups, with flags: the default model inputs
print(f"Default model inputs: {len(feature_cols)} columns (17 features + 17 missing flags)")
feature_table_df.style.format({"pct_blank": "{:.1f}"}).hide(axis="index")

Default model inputs: 34 columns (17 features + 17 missing flags)


group,feature,kind,missing_flag,pct_blank
family,ESCS,numeric,ESCS_missing,3.9
family,PAREDINT,numeric,PAREDINT_missing,4.3
family,HISEI,numeric,HISEI_missing,14.6
family,HOMEPOS,numeric,HOMEPOS_missing,2.9
maths_attitudes,ANXMAT,numeric,ANXMAT_missing,22.9
maths_attitudes,MATHEFF,numeric,MATHEFF_missing,21.1
maths_attitudes,MATHEF21,numeric,MATHEF21_missing,22.4
classroom,DISCLIM,numeric,DISCLIM_missing,16.9
classroom,TEACHSUP,numeric,TEACHSUP_missing,18.0
climate_resources,BELONG,numeric,BELONG_missing,9.2


In [7]:
without_classroom_cols = prep.get_feature_cols([group for group in prep.GROUPS if group != "classroom"])
print("Dropping the classroom group removes:", sorted(set(feature_cols) - set(without_classroom_cols)))
print("Family group alone:", prep.get_feature_cols(["family"]))

# The pipeline refuses columns that would leak the answer (other test scores, targets, weights, IDs).
try:
    prep.make_preprocessor(feature_cols + ["PV1READ", "W_FSTUWT"])
except ValueError as error:
    print("Refused:", error)

Dropping the classroom group removes: ['DISCLIM', 'DISCLIM_missing', 'TEACHSUP', 'TEACHSUP_missing']
Family group alone: ['ESCS', 'PAREDINT', 'HISEI', 'HOMEPOS', 'ESCS_missing', 'PAREDINT_missing', 'HISEI_missing', 'HOMEPOS_missing']
Refused: these columns must never be model inputs (leakage): ['PV1READ', 'W_FSTUWT']


## 5. The preprocessing pipeline

`prep.make_preprocessor(feature_cols)` sends each kind of column through the steps that suit it:

| Columns | Step | What it learns from the training schools |
|---|---|---|
| 16 numeric features | 1. **Clip**: cap values below the 1st and above the 99th percentile, so a few extreme values can't pull the model around | the two cut points per feature |
| | 2. **Fill blanks** with the median (the middle value) | the median per feature |
| | 3. **Scale** to average 0 and spread (standard deviation) 1, so features measured on big scales don't drown out small ones | the mean and standard deviation per feature |
| `SCHL_PRIVATE` (0/1) | **Fill blanks** with the most common value | the most common value |
| 17 missing flags | passed through unchanged | nothing |
| `CNT` (only if the group decides to use it) | **One-hot encode**: one 0/1 column per system | the list of systems |

Every step has two halves. **`fit`** learns its numbers and is only ever run on the training rows.
**`transform`** applies those same numbers to any rows, including the test rows. A scikit-learn `Pipeline` does this automatically:
`pipeline.fit(train)` fits every step on the training rows; `pipeline.predict(test)` only transforms.

Scaling matters for logistic regression, k-NN and neural networks. Decision trees and random forests ignore it, so it's harmless for them.
The numbers are learned without survey weights (scikit-learn's imputer can't use weights). That only moves the fill-in values and
cut points slightly, and the weights are used where they matter: fitting and scoring the model (section 7).

In [8]:
preprocessor = prep.make_preprocessor(feature_cols)
preprocessor   # click the boxes to see each step's settings

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('binary', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per tran

## 6. Proof that the pipeline only learns from the training schools

Take round 0: fold 0 is the test set, folds 1-4 are training. Fit the preprocessor on the training rows only, then compare the
**medians it learned** with medians worked out by hand. They should equal the training medians exactly,
and differ from the medians of the whole table and of the test schools. If test rows had leaked in, they wouldn't.

In [9]:
is_test = fold_by_student == 0
train_df, test_df = df[~is_test], df[is_test]
preprocessor = prep.make_preprocessor(feature_cols).fit(train_df[feature_cols])   # fit on training rows only

numeric_pipeline = preprocessor.named_transformers_["numeric"]
numeric_cols = list(numeric_pipeline.feature_names_in_)
clipper = numeric_pipeline.named_steps["clip"]
learned_df = pd.DataFrame({
    "learned_median": numeric_pipeline.named_steps["fill_blanks"].statistics_,
    "training_median": train_df[numeric_cols].median(),     # worked out by hand
    "whole_table_median": df[numeric_cols].median(),
    "test_median": test_df[numeric_cols].median(),
    "learned_low_cap": clipper.lower_bounds_,
    "learned_high_cap": clipper.upper_bounds_,
    "n_training_values_capped": ((train_df[numeric_cols] < clipper.lower_bounds_) | (train_df[numeric_cols] > clipper.upper_bounds_)).sum(),
}, index=numeric_cols)
learned_df.to_csv(OUTPUT_DIR / "learned_vs_whole_table.csv")

assert np.allclose(learned_df["learned_median"], learned_df["training_median"])
print("The learned medians equal the training medians.")
print(f"They differ from the whole-table medians for {(~np.isclose(learned_df['learned_median'], learned_df['whole_table_median'])).sum()} "
      f"of {len(numeric_cols)} features, and from the test medians for "
      f"{(~np.isclose(learned_df['learned_median'], learned_df['test_median'])).sum()}.")
learned_df

The learned medians equal the training medians.
They differ from the whole-table medians for 7 of 16 features, and from the test medians for 10.


,learned_median,training_median,whole_table_median,test_median,learned_low_cap,learned_high_cap,n_training_values_capped
ESCS,-0.018,-0.018,-0.022,-0.039,-3.728,1.583,952
PAREDINT,16.000,16.000,16.000,16.000,3.000,16.000,0
HISEI,65.010,65.010,65.010,65.120,17.000,88.700,396
HOMEPOS,-0.733,-0.733,-0.739,-0.764,-4.021,2.137,963
ANXMAT,0.193,0.193,0.193,0.193,-2.394,2.635,0
MATHEFF,-0.435,-0.435,-0.431,-0.414,-3.452,2.274,751
MATHEF21,0.196,0.196,0.202,0.238,-2.283,2.594,766
DISCLIM,-0.078,-0.078,-0.078,-0.072,-2.494,1.849,599
TEACHSUP,0.147,0.147,0.147,0.147,-2.909,1.556,0
BELONG,-0.338,-0.338,-0.338,-0.338,-1.920,2.747,850


The same check for scaling. After the transform, the **training** rows have mean exactly 0 and spread exactly 1,
because those are the rows the scaler learned from. The **test** rows come close but not exactly,
because they were transformed with the training numbers. And no blanks are left in either.

In [10]:
transformed_train_df = preprocessor.transform(train_df[feature_cols])
transformed_test_df = preprocessor.transform(test_df[feature_cols])

scale_check_df = pd.DataFrame({
    "train_mean": transformed_train_df[numeric_cols].mean(), "train_sd": transformed_train_df[numeric_cols].std(ddof=0),
    "test_mean": transformed_test_df[numeric_cols].mean(), "test_sd": transformed_test_df[numeric_cols].std(ddof=0),
})
assert np.allclose(scale_check_df["train_mean"], 0, atol=1e-9) and np.allclose(scale_check_df["train_sd"], 1, atol=1e-9)
assert transformed_train_df.notna().all().all() and transformed_test_df.notna().all().all()
print("Training rows: mean 0 and spread 1 exactly. No blanks left in training or test rows.")
print(f"Model input table: {transformed_test_df.shape[1]} columns")
scale_check_df

Training rows: mean 0 and spread 1 exactly. No blanks left in training or test rows.
Model input table: 34 columns


,train_mean,train_sd,test_mean,test_sd
ESCS,0.000,1.000,-0.015,0.999
PAREDINT,-0.000,1.000,-0.004,1.012
HISEI,0.000,1.000,0.012,0.992
HOMEPOS,0.000,1.000,-0.035,0.991
ANXMAT,0.000,1.000,-0.008,0.990
MATHEFF,-0.000,1.000,0.020,0.995
MATHEF21,-0.000,1.000,0.016,1.000
DISCLIM,0.000,1.000,0.006,1.000
TEACHSUP,0.000,1.000,0.011,0.986
BELONG,-0.000,1.000,-0.003,1.002


**So what.** Every learned number came from the training schools only. The learned medians match the training medians exactly.
They differ from the whole-table medians for 7 of the 16 numeric features and from the test-school medians for 10.
The rest coincide because those features take a limited number of values (years of schooling, class-size bands,
indices built from a few questions), so the middle value is the same whichever rows you use.
After scaling, the training rows have mean 0 and spread 1 exactly, and the test rows land near those values but not on them, as they should.
Clipping capped up to 2% of a feature's training values (1% at each end, by design), and none at all in the six features
whose 1st and 99th percentiles are already their lowest and highest values.

## 7. Weights

PISA doesn't sample every country in proportion to its size, so each student carries a **weight**: how many real 15-year-olds they stand for.

- `W_FSTUWT` (final student weight): the population view. Pooled, the big systems dominate.
- `SENWT` (senate weight): the same weights, rescaled so that **each system adds up to 5,000**. Pooled, each system counts equally.

In [11]:
weight_df = (df.groupby("CNT")
             .agg(n_students=("CNTSTUID", "size"), W_FSTUWT=("W_FSTUWT", "sum"), SENWT=("SENWT", "sum"))
             .loc[prep.SYSTEMS].rename(index=NAME_BY_SYSTEM))
weight_df["pct_of_sample"] = 100 * weight_df["n_students"] / weight_df["n_students"].sum()
weight_df["pct_of_W_FSTUWT"] = 100 * weight_df["W_FSTUWT"] / weight_df["W_FSTUWT"].sum()
weight_df["pct_of_SENWT"] = 100 * weight_df["SENWT"] / weight_df["SENWT"].sum()

# Inside one system, SENWT is W_FSTUWT times a constant, so the two give the same results for a single system.
# (SENWT is stored rounded to 5 decimals, so the ratio is constant only up to that rounding.)
senwt_ratio_by_student = df["SENWT"] / df["W_FSTUWT"]
ratio_variation_by_system = senwt_ratio_by_student.groupby(df["CNT"]).agg(lambda ratios: ratios.std() / ratios.mean())
assert (ratio_variation_by_system < 1e-3).all()
print("Inside each system SENWT is a constant multiple of W_FSTUWT (up to rounding).")

weight_df.style.format({"n_students": "{:,}", "W_FSTUWT": "{:,.0f}", "SENWT": "{:,.0f}", "pct_of_sample": "{:.1f}",
                        "pct_of_W_FSTUWT": "{:.1f}", "pct_of_SENWT": "{:.1f}"})

Inside each system SENWT is a constant multiple of W_FSTUWT (up to rounding).


,n_students,W_FSTUWT,SENWT,pct_of_sample,pct_of_W_FSTUWT,pct_of_SENWT
CNT,,,,,,
UAE,"24,600","60,765","5,000",39.8,5.6,16.7
Saudi Arabia,"6,928","317,452","5,000",11.2,29.3,16.7
Qatar,"7,676","18,348","5,000",12.4,1.7,16.7
Jordan,"7,799","144,269","5,000",12.6,13.3,16.7
Morocco,"6,867","454,986","5,000",11.1,42.0,16.7
Palestine,"7,905","88,383","5,000",12.8,8.2,16.7


**How Part B uses them.**

- **Pooled models** (all six systems together) use `prep.POOLED_WEIGHT_COL`, set to `SENWT` for now, so that the UAE
  (40% of the sample) doesn't dominate and Morocco and Saudi Arabia (71% of the population) don't either. This is a team decision (section 10).
- **One-system models** use `W_FSTUWT`. `prep.get_weight_col()` picks the right one from the rows you give it.
- `prep.get_sample_weights()` rescales the weights to **average 1** before they go into a model.
  That doesn't change which students count more. It keeps the total weight equal to the number of students, so regularisation settings
  (`C` in logistic regression, `alpha` in ridge or an MLP) mean the same thing as without weights. Raw `W_FSTUWT` adds up to millions,
  which would quietly switch regularisation off.
- The same weights are used to **score** the model on the test fold (weighted AUC, weighted RMSE), so the scores describe the population, not the sample.

In [12]:
sample_weights = prep.get_sample_weights(df)
print(f"Pooled weight column: {prep.get_weight_col(df)}; one system: {prep.get_weight_col(df[df['CNT'] == 'MAR'])}")
print(f"Rescaled weights: mean {sample_weights.mean():.3f}, total {sample_weights.sum():,.0f} (= {len(df):,} students)")

Pooled weight column: SENWT; one system: W_FSTUWT
Rescaled weights: mean 1.000, total 61,775 (= 61,775 students)


## 8. Smoke test: does a simple model run end to end?

A **smoke test** checks that the plumbing works, not how good the model is. **These are not our results.**
The real results (T19 onwards) train once per plausible value, combine the 10 with Rubin's rules, and add standard errors from the replicate weights.

Here: weighted logistic regression predicting `BELOW_L2_PV1` (below Level 2 on the first plausible value), 5 folds, scored by weighted AUC on the
test schools, for three sets of inputs:

- **family group alone**: the pitch's pilot used home background alone and got AUC 0.749, so this should land close to that;
- **all 5 groups**: the default inputs;
- **all 5 groups without the missing flags**: to see whether the flags help.

Plus a ridge regression predicting `PV1MATH` from all 5 groups, scored by weighted RMSE (typical error in score points) and R² (share of variation explained).
The project plan's leakage alarm: any AUC above 0.90 means something leaked. The notebook stops if that happens.

In [13]:
TARGET_COL = prep.get_target_col("classification", pv=1)       # "BELOW_L2_PV1"
SCORE_COL = prep.get_target_col("regression", pv=1)             # "PV1MATH"
INPUT_COLS_BY_SET = {
    "family group alone": prep.get_feature_cols(["family"]),
    "all 5 groups": prep.get_feature_cols(),
    "all 5 groups, no missing flags": prep.get_feature_cols(with_missing_flags=False),
}

smoke_rows = []                                                 # one dict per input set and fold
out_of_fold_prob_by_student = pd.Series(np.nan, index=df.index) # each student's prediction from the model that never saw their school
for fold in range(prep.N_FOLDS):
    is_test = fold_by_student == fold
    train_df, test_df = df[~is_test], df[is_test]
    train_weights, test_weights = prep.get_sample_weights(train_df), prep.get_sample_weights(test_df)

    for input_set, input_cols in INPUT_COLS_BY_SET.items():
        classifier = prep.make_model_pipeline(LogisticRegression(max_iter=2000), input_cols)
        classifier.fit(train_df[input_cols], train_df[TARGET_COL], model__sample_weight=train_weights)
        predicted_probs = classifier.predict_proba(test_df[input_cols])[:, 1]
        smoke_rows.append({"inputs": input_set, "fold": fold, "metric": "AUC",
                           "value": roc_auc_score(test_df[TARGET_COL], predicted_probs, sample_weight=test_weights)})
        if input_set == "all 5 groups":
            out_of_fold_prob_by_student[is_test] = predicted_probs

    regressor = prep.make_model_pipeline(Ridge(alpha=1.0), feature_cols)
    regressor.fit(train_df[feature_cols], train_df[SCORE_COL], model__sample_weight=train_weights)
    predicted_scores = regressor.predict(test_df[feature_cols])
    smoke_rows.append({"inputs": "all 5 groups", "fold": fold, "metric": "RMSE (ridge)",
                       "value": root_mean_squared_error(test_df[SCORE_COL], predicted_scores, sample_weight=test_weights)})
    smoke_rows.append({"inputs": "all 5 groups", "fold": fold, "metric": "R² (ridge)",
                       "value": r2_score(test_df[SCORE_COL], predicted_scores, sample_weight=test_weights)})

smoke_df = pd.DataFrame(smoke_rows)
smoke_df.to_csv(OUTPUT_DIR / "smoke_test.csv", index=False)
assert smoke_df.query("metric == 'AUC'")["value"].max() < 0.90, "AUC above 0.90: check for leakage"

smoke_table_df = smoke_df.pivot_table(index=["metric", "inputs"], columns="fold", values="value", sort=False)
smoke_table_df.columns = [f"fold {fold}" for fold in smoke_table_df.columns]
smoke_table_df["mean"] = smoke_table_df.mean(axis=1)
smoke_table_df

fold 0  fold 1  fold 2  fold 3  fold 4   mean
metric       inputs                                                                       
AUC          family group alone               0.746   0.747   0.734   0.737   0.724  0.738
             all 5 groups                     0.814   0.809   0.794   0.804   0.798  0.804
             all 5 groups, no missing flags   0.796   0.791   0.773   0.789   0.778  0.785
RMSE (ridge) all 5 groups                    64.855  63.974  64.311  65.257  63.757 64.431
R² (ridge)   all 5 groups                     0.381   0.363   0.320   0.373   0.368  0.361

The same "all 5 groups" model, scored separately for each system. Each student's prediction comes from the round in which their school was in the test fold,
so every prediction is for an unseen school. Weighted with `W_FSTUWT`.

In [14]:
auc_by_system = pd.Series({
    NAME_BY_SYSTEM[system]: roc_auc_score(df.loc[df["CNT"] == system, TARGET_COL],
                                          out_of_fold_prob_by_student[df["CNT"] == system],
                                          sample_weight=df.loc[df["CNT"] == system, "W_FSTUWT"])
    for system in prep.SYSTEMS}, name="AUC (all 5 groups, PV1)")
auc_by_system.to_frame()

,"AUC (all 5 groups, PV1)"
UAE,0.803
Saudi Arabia,0.707
Qatar,0.788
Jordan,0.771
Morocco,0.792
Palestine,0.765


**So what.** The plumbing works end to end.

- The **family group alone** averages AUC 0.738, close to the pitch's pilot (0.749, also home background with whole schools held out).
  The small gap can come from a different split or weights.
- **All 5 groups** average 0.804 and beat the family group in every fold.
- **Without the missing flags** the AUC drops to 0.785, which supports the EDA finding that blank answers carry information.
- **Ridge regression** misses a student's maths score by about 64 points (RMSE) and explains 36% of the variation (R²).
- No AUC is anywhere near the 0.90 leakage alarm.

Per system, AUC ranges from 0.71 (Saudi Arabia) to 0.80 (UAE), no higher than the pooled 0.80.
That's expected: in a pooled score, part of the model's skill is telling lower-scoring systems from higher-scoring ones.
The per-system numbers are the harder test, and in this first run only Saudi Arabia is below the pitch's 0.75 target.
Again: one plausible value, one untuned model. These are not results.

## 9. How to use Part B in your own notebook

Copy this into any notebook in the repo (baselines, neural networks, ablations). Swap in your own model:

```python
import sys
from pathlib import Path
ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "requirements.txt").exists())
sys.path.insert(0, str(ROOT / "02_data-analysis"))
import pisa_prep as prep

df = prep.load_clean_table()
fold_by_student = prep.get_fold_by_student(df)          # the frozen folds
feature_cols = prep.get_feature_cols()                   # or prep.get_feature_cols(["family", "classroom"]) for an ablation

for pv in range(1, 11):                                  # all 10 plausible values
    target_col = prep.get_target_col("classification", pv)
    for fold in range(prep.N_FOLDS):
        is_test = fold_by_student == fold
        train_df, test_df = df[~is_test], df[is_test]
        model = prep.make_model_pipeline(LogisticRegression(max_iter=2000), feature_cols)
        model.fit(train_df[feature_cols], train_df[target_col],
                  model__sample_weight=prep.get_sample_weights(train_df))
        predicted_probs = model.predict_proba(test_df[feature_cols])[:, 1]
        # score with sample_weight=prep.get_sample_weights(test_df)
```

What `prep` gives you:

| Name | What it is |
|---|---|
| `load_clean_table()` | the Part A table |
| `get_fold_by_student(df)` | the test fold (0-4) of each student, from the frozen `folds.csv` |
| `get_feature_cols(groups)` | model input columns for the chosen factor groups, with their missing flags |
| `make_preprocessor(feature_cols)` | clip, fill blanks, scale (section 5); `scale=False` for trees if you like |
| `make_model_pipeline(model, feature_cols)` | preprocessing + your model in one object |
| `get_target_col(task, pv)` | `"PV{pv}MATH"` or `"BELOW_L2_PV{pv}"` |
| `get_sample_weights(rows_df)` | the right survey weight for those rows, rescaled to average 1 |
| `make_inner_cv()` | school-grouped folds *inside* a training set, for tuning hyperparameters (pass `groups=train_df["CNTSCHID"]`) |
| `check_no_leakage(cols)` | stops with an error if any column must never be a model input |

Three rules:

1. **Never fit anything on the whole table.** Fit only on `train_df`. The pipeline does this for you if all preprocessing is inside it.
2. **Tune inside the training folds.** Hyperparameter searches use `make_inner_cv()` on `train_df`, never the test fold.
3. **Don't make your own folds** or edit `folds.csv`. If the folds must change, change `pisa_prep.py`, set `REBUILD_FOLDS = True` here, re-run, and tell everyone.

Some models can't take `sample_weight` (k-NN, for example). Say so in the report when you use one.

## 10. Settings the group still has to agree on

These four are open team decisions. Each has a default so work can go ahead, and each is one line at the top of `pisa_prep.py`.
Changing a line there changes it for everyone.

| Setting in `pisa_prep.py` | Default | Alternative | Why the default |
|---|---|---|---|
| `POOLED_WEIGHT_COL` | `"SENWT"`: each system counts equally | `"W_FSTUWT"`: each system counts by its population (Morocco and Saudi Arabia = 71%) | The project compares six systems, and the EDA pooled with `SENWT`. With `W_FSTUWT`, a pooled model is mostly a Morocco + Saudi Arabia model. |
| `USE_SYSTEM_AS_FEATURE` | `False` | `True`: add `CNT` as 6 one-hot columns | The pitch's 17 predictors don't include it. Adding it can raise AUC just by learning each system's average, which says little about students. Worth a comparison run. |
| `FAMILY_AS_ESCS_ONLY` | `False`: ESCS + its 3 parts | `True`: ESCS alone | Keeps the pitch's 17 predictors. ESCS is built from the other three (correlation about 0.8), which matters for reading linear-model coefficients, not for prediction. |
| `CLIP_PERCENTILES` | `(1.0, 99.0)` | `None` (no clipping), or wider limits | Several indices have long tails (EDA section 5). Clipping 2% of values protects linear models and MLPs; trees are unaffected. |

**Next (not part of Part B):** T19, the evaluation code: run each model for all 10 plausible values, combine the 10 results with Rubin's rules,
and get standard errors from the 80 replicate weights.